In [1]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Parameters
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 123

# Enable Mixed Precision (saves VRAM, speeds up training)
mixed_precision.set_global_policy("mixed_float16")

2025-09-06 20:15:23.933780: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757189724.246237      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757189724.339669      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
BASE_DIR = "/kaggle/input/skin-diseases/kaggle"

train_dir = os.path.join(BASE_DIR, "train")
val_dir   = os.path.join(BASE_DIR, "val")
test_dir  = os.path.join(BASE_DIR, "test")

AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    seed=SEED,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    seed=SEED,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    seed=SEED,
    shuffle=False
)


class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

import json

# Save the class names list (not indices) in the correct order
with open("class_names.json", "w") as f:
    json.dump(class_names, f)

Found 30909 files belonging to 6 classes.


I0000 00:00:1757189766.003714      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1757189766.004399      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 3923 files belonging to 6 classes.
Found 3928 files belonging to 6 classes.
Classes: ['1. Enfeksiyonel', '2. Ekzama', '3. Akne', '4. Pigment', '5. Benign', '6. Malign']


In [3]:
AUTOTUNE = tf.data.AUTOTUNE

def prepare(ds, shuffle=False):
    if shuffle:
        ds = ds.shuffle(1000, seed=SEED)
    ds = ds.prefetch(buffer_size=AUTOTUNE)   # async pipeline
    return ds

train_ds = prepare(train_ds, shuffle=True)
val_ds   = prepare(val_ds)
test_ds  = prepare(test_ds)

In [4]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [5]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    input_shape=IMAGE_SIZE + (3,),
    weights="imagenet"
)
base_model.trainable = False   # freeze base model initially

inputs = layers.Input(shape=IMAGE_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)  # normalization
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)  
# dtype="float32" ensures stable outputs in mixed precision

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cast_1 (Cast)                   │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cast_2 (Cast)                   │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │         7,686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,057,257 (15.48 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [6]:
checkpoint = ModelCheckpoint(
    "/kaggle/working/best_model.keras",
    save_best_only=True,
    monitor="val_accuracy",
    mode="max",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    verbose=1
)

callbacks = [checkpoint, early_stop, reduce_lr]

In [7]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20


I0000 00:00:1757190075.795418     105 cuda_dnn.cc:529] Loaded cuDNN version 90300


1932/1932 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.4254 - loss: 1.3774
Epoch 1: val_accuracy improved from -inf to 0.61382, saving model to /kaggle/working/best_model.keras
1932/1932 ━━━━━━━━━━━━━━━━━━━━ 152s 45ms/step - accuracy: 0.4254 - loss: 1.3773 - val_accuracy: 0.6138 - val_loss: 0.9777 - learning_rate: 1.0000e-04
Epoch 2/20
1932/1932 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.5946 - loss: 1.0107
Epoch 2: val_accuracy improved from 0.61382 to 0.63727, saving model to /kaggle/working/best_model.keras
1932/1932 ━━━━━━━━━━━━━━━━━━━━ 111s 42ms/step - accuracy: 0.5946 - loss: 1.0107 - val_accuracy: 0.6373 - val_loss: 0.9221 - learning_rate: 1.0000e-04
Epoch 3/20
1931/1932 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.6201 - loss: 0.9430
Epoch 3: val_accuracy improved from 0.63727 to 0.65180, saving model to /kaggle/working/best_model.keras
1932/1932 ━━━━━━━━━━━━━━━━━━━━ 107s 42ms/step - accuracy: 0.6201 - loss: 0.9430 - val_accuracy: 0.6518 - val_loss: 0.8913 - learning

In [8]:
base_model.trainable = True

# Recompile with lower LR
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20
1932/1932 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - accuracy: 0.4927 - loss: 1.2769
Epoch 1: val_accuracy did not improve from 0.69615
1932/1932 ━━━━━━━━━━━━━━━━━━━━ 422s 174ms/step - accuracy: 0.4928 - loss: 1.2768 - val_accuracy: 0.6503 - val_loss: 0.9193 - learning_rate: 1.0000e-05
Epoch 2/20
1932/1932 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - accuracy: 0.6324 - loss: 0.9346
Epoch 2: val_accuracy did not improve from 0.69615
1932/1932 ━━━━━━━━━━━━━━━━━━━━ 351s 169ms/step - accuracy: 0.6324 - loss: 0.9346 - val_accuracy: 0.6750 - val_loss: 0.8449 - learning_rate: 1.0000e-05
Epoch 3/20
1932/1932 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step - accuracy: 0.6754 - loss: 0.8305
Epoch 3: val_accuracy improved from 0.69615 to 0.69641, saving model to /kaggle/working/best_model.keras
1932/1932 ━━━━━━━━━━━━━━━━━━━━ 353s 170ms/step - accuracy: 0.6754 - loss: 0.8305 - val_accuracy: 0.6964 - val_loss: 0.7961 - learning_rate: 1.0000e-05
Epoch 4/20
1932/1932 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - accuracy: 0.

In [9]:
# 8. Reload Best Model (after restart)
# ============================
# If Kaggle disconnects or after training ends
import tensorflow as tf
best_model = tf.keras.models.load_model("best_model.keras")

ValueError: File not found: filepath=best_model.keras. Please ensure the file is an accessible `.keras` zip file.

In [7]:
import numpy as np

# Get true labels and predictions for TRAIN
y_true_train = np.concatenate([y.numpy() for x,y in train_ds], axis=0).argmax(axis=1)
y_pred_train = np.argmax(best_model.predict(train_ds), axis=1)

# Get true labels and predictions for TEST
y_true_test = np.concatenate([y.numpy() for x,y in test_ds], axis=0).argmax(axis=1)
y_pred_test = np.argmax(best_model.predict(test_ds), axis=1)

I0000 00:00:1756370649.466899    1583 cuda_dnn.cc:529] Loaded cuDNN version 90300


1833/1833 ━━━━━━━━━━━━━━━━━━━━ 105s 36ms/step
230/230 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step


In [10]:
ls -lh

total 79M
-rw-r--r-- 1 root root 79M Aug 28 08:29 best_model.keras


In [11]:
import shutil

shutil.move("best_model.keras", "/kaggle/working/best_model.keras")

'/kaggle/working/best_model.keras'

In [13]:
from IPython.display import FileLink

# Generate a clickable download link
FileLink(r'best_model.keras')


/kaggle/working/best_model.keras

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

print("===== TRAIN METRICS =====")
print(classification_report(y_true_train, y_pred_train, target_names=train_ds.class_names))

print("===== TEST METRICS =====")
print(classification_report(y_true_test, y_pred_test, target_names=test_ds.class_names))

# Or separately if you just want macro averages:
train_precision = precision_score(y_true_train, y_pred_train, average='macro')
train_recall    = recall_score(y_true_train, y_pred_train, average='macro')
train_f1        = f1_score(y_true_train, y_pred_train, average='macro')

test_precision = precision_score(y_true_test, y_pred_test, average='macro')
test_recall    = recall_score(y_true_test, y_pred_test, average='macro')
test_f1        = f1_score(y_true_test, y_pred_test, average='macro')

print(f"Train -> Precision: {train_precision:.4f}, Recall: {train_recall:.4f}, F1: {train_f1:.4f}")
print(f"Test  -> Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, F1: {test_f1:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import (
    classification_report,
    precision_score, recall_score, f1_score,
    confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt

# ==== 2. Classification Reports ====
print("===== TRAIN METRICS =====")
print(classification_report(y_true_train, y_pred_train, target_names=train_ds.class_names))

print("===== TEST METRICS =====")
print(classification_report(y_true_test, y_pred_test, target_names=test_ds.class_names))

# Macro averages
def macro_metrics(y_true, y_pred, split):
    prec = precision_score(y_true, y_pred, average='macro')
    rec  = recall_score(y_true, y_pred, average='macro')
    f1   = f1_score(y_true, y_pred, average='macro')
    print(f"{split} -> Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")

macro_metrics(y_true_train, y_pred_train, "TRAIN")
macro_metrics(y_true_test, y_pred_test, "TEST")

# ==== 3. Confusion Matrices ====
def plot_confusion(y_true, y_pred, classes, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=False, fmt="d", cmap="Blues", 
                xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

plot_confusion(y_true_train, y_pred_train, train_ds.class_names, "Confusion Matrix - TRAIN")
plot_confusion(y_true_test, y_pred_test, test_ds.class_names, "Confusion Matrix - TEST")
